# R2AI · embed bge-m3 toàn corpus (chạy thẳng trên GPU Colab)

Notebook này đọc các phần chunk trên GCS, tính bge-m3 dense + sparse rồi ghi shard trở lại GCS. Nó gọi đúng hàm `rag.run_embed` của `3-ingest`.

**Điều kiện trước:** VM đã chạy `rag chunk` xong, tức `OUT/rag/chunks/<lang>/*.parquet` đã có. Code đã được đóng gói lên `CODE` (xem `3-ingest/colab/pack.py`).

**Chạy song song nhiều phiên** (Colab/Kaggle của từng thành viên):
- Mỗi phiên đặt `SHARD` riêng, ví dụ 3 phiên dùng `"0/3"`, `"1/3"`, `"2/3"`. Hai phiên không bao giờ embed trùng một phần.
- Bị ngắt (hết giờ, rớt kết nối) thì **chạy lại từ đầu với cùng `SHARD`**. Phần nào đã có shard sẽ được bỏ qua. Chỉ mất phần đang chạy dở (tối đa khoảng 1 giờ).
- Khi mọi phiên xong, cell cuối báo `todo = 0` cho toàn bộ. Lúc đó chạy `rag index` trên VM.

Runtime: **T4 GPU** (Runtime → Change runtime type). Giữ tab mở để Colab không ngắt vì idle.

In [ ]:
OUT = "gs://aiguru_crawler/ingest/v1"           # ingest root written by `rag chunk`
CODE = "gs://aiguru_crawler/code/r2ai_ingest.zip"  # bundle from 3-ingest/colab/pack.py
SHARD = "0/1"                                     # this session's share: "i/n" for n parallel sessions
BATCH, MAX_LEN = 32, 2048                         # T4 fp16; lower BATCH if CUDA OOM
LANGS = None                                      # e.g. ["en"] to embed one language first

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
DEVICES = [f"cuda:{k}" for k in range(torch.cuda.device_count())]
print(DEVICES, torch.cuda.get_device_name(0))

In [ ]:
from google.colab import auth
auth.authenticate_user()  # your Google account: needs read/write on the bucket

In [ ]:
!pip -q install FlagEmbedding==1.4.2
!python -c "import gcsfs" 2>/dev/null || pip -q install gcsfs
!rm -rf /content/r2ai && mkdir -p /content/r2ai && gcloud storage cp {CODE} /content/r2ai/code.zip && cd /content/r2ai && unzip -q -o code.zip && ls

In [ ]:
import logging, os, sys, time, zlib
sys.path.insert(0, "/content/r2ai")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
for noisy in ("gcsfs", "urllib3", "google", "fsspec"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
from extract import storage
from ingest import rag

I, N = (int(x) for x in SHARD.split("/"))
assert 0 <= I < N, SHARD

def status(shard=None):
    """(parts, done, todo) over all parts, or over one session's share."""
    parts = storage.glob(storage.join(OUT, "rag", "chunks", "*", "*.parquet"))
    rows = []
    for p in parts:
        lang, name = p.split("/")[-2], p.split("/")[-1][:-len(".parquet")]
        if LANGS and lang not in LANGS:
            continue
        if shard and zlib.crc32(f"{lang}/{name}".encode()) % shard[1] != shard[0]:
            continue
        d = storage.join(OUT, "rag", "dense", "bge-m3", lang, name) + ".ids.parquet"
        s = storage.join(OUT, "rag", "sparse", "bge-m3", lang, name) + ".ids.parquet"
        rows.append(storage.exists(d) and storage.exists(s))
    return len(rows), sum(rows), len(rows) - sum(rows)

n, done, todo = status((I, N))
print(f"shard {SHARD}: {n} parts, {done} done, {todo} to embed")
assert n, f"no chunk parts under {OUT}/rag/chunks: run `rag chunk` first"


In [ ]:
enc = rag.BgeM3Encoder("BAAI/bge-m3", DEVICES, BATCH, MAX_LEN)
v, lex = enc.encode(["Đau đầu kéo dài có nguy hiểm không?", "高血压患者能喝咖啡吗", "metformin and lactic acidosis"])
assert v.shape == (3, 1024) and all(lex), (v.shape, lex)
print("smoke ok: dense", v.shape, "| sparse tokens", [len(x) for x in lex])

In [ ]:
t0 = time.time()
res = rag.run_embed(OUT, enc, langs=LANGS, workdir="/content/work", shard=(I, N))
print(res, f"{(time.time() - t0) / 3600:.2f} h")
print("this shard:", status((I, N)))

In [ ]:
# all sessions together: todo == 0 -> run `python -m ingest rag index --out OUT` on the VM
print("all shards (parts, done, todo):", status())